In [ ]:
# CELL 0: INSTALL & IMPORTS

# !pip install -q tensorflow keras scikit-learn scipy matplotlib seaborn

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import (
    LeaveOneGroupOut, GroupShuffleSplit, train_test_split
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy import stats as sp_stats

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPooling2D, Flatten,
    Dropout, LSTM, Bidirectional, concatenate, Layer
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')
print(f"TensorFlow : {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")



In [ ]:
# CELL 1: MOUNT GOOGLE DRIVE & SET PATHS

from google.colab import drive
import os
drive.mount('/content/drive')

# =============================================================
# Updated path to the 'final' folder containing train.csv
# =============================================================
SWELL_DIR = "/content/drive/MyDrive/SWELL-KW/data/final"

RESULTS_DIR = "/content/drive/MyDrive/SWELL-KW/HABC_Results_SWELL"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Directly setting csv_file to train.csv so your downstream code doesn't break
csv_file = os.path.join(SWELL_DIR, "train.csv")

if not os.path.exists(csv_file):
    raise FileNotFoundError(f"train.csv nahi mili in {SWELL_DIR}. Ek baar path confirm karein.")

print(f"Using dataset : {csv_file}")
print(f"Results dir   : {RESULTS_DIR}")

In [ ]:
# CELL 2: ATTENTION LAYER + ALL MODEL BUILDERS

class AttentionLayer(Layer):
    """Temporal attention: learns which time steps matter most."""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight",
                                 shape=(input_shape[-1], 1), initializer="normal")
        self.b = self.add_weight(name="att_bias",
                                 shape=(input_shape[1], 1), initializer="zeros")
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        et = K.squeeze(K.tanh(K.dot(x, self.W) + self.b), axis=-1)
        at = K.softmax(et)
        at = K.expand_dims(at, axis=-1)
        return K.sum(x * at, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        return super(AttentionLayer, self).get_config()


def build_habc_model(num_classes, time_steps, feature_dim, lr=0.001):
    """FULL HABC: CNN + BiLSTM + Attention."""
    cnn_inp = Input(shape=(1, time_steps, feature_dim), name='cnn_input')
    c = Conv2D(6, (3,3), padding='same', activation='relu')(cnn_inp)
    c = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(c)
    c = Flatten()(c)

    lstm_inp = Input(shape=(time_steps, feature_dim), name='lstm_input')
    l = Bidirectional(LSTM(64, return_sequences=True))(lstm_inp)
    l = AttentionLayer()(l)

    merged = concatenate([c, l])
    x = Dense(64, activation='relu')(merged)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)

    m = Model(inputs=[cnn_inp, lstm_inp], outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              metrics=['accuracy'])
    return m


def build_cnn_only(num_classes, time_steps, feature_dim, lr=0.001):
    inp = Input(shape=(1, time_steps, feature_dim))
    x = Conv2D(6, (3,3), padding='same', activation='relu')(inp)
    x = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(x)
    x = Flatten()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              metrics=['accuracy'])
    return m


def build_bilstm_only(num_classes, time_steps, feature_dim, lr=0.001):
    inp = Input(shape=(time_steps, feature_dim))
    x = Bidirectional(LSTM(64, return_sequences=False))(inp)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              metrics=['accuracy'])
    return m


def build_cnn_bilstm_no_attention(num_classes, time_steps, feature_dim, lr=0.001):
    cnn_inp = Input(shape=(1, time_steps, feature_dim))
    c = Conv2D(6, (3,3), padding='same', activation='relu')(cnn_inp)
    c = MaxPooling2D(pool_size=(1,2), strides=(1,2), padding='same')(c)
    c = Flatten()(c)
    lstm_inp = Input(shape=(time_steps, feature_dim))
    l = Bidirectional(LSTM(64, return_sequences=False))(lstm_inp)
    merged = concatenate([c, l])
    x = Dense(64, activation='relu')(merged)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=[cnn_inp, lstm_inp], outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              metrics=['accuracy'])
    return m


def build_bilstm_attention_only(num_classes, time_steps, feature_dim, lr=0.001):
    inp = Input(shape=(time_steps, feature_dim))
    x = Bidirectional(LSTM(64, return_sequences=True))(inp)
    x = AttentionLayer()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    m = Model(inputs=inp, outputs=out)
    m.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
              metrics=['accuracy'])
    return m


In [ ]:
# CELL 3: LOAD & PREPROCESS SWELL-KW DATA

def load_swell_kw(csv_path):
    """
    Load SWELL-KW HRV dataset from Kaggle CSV.

    The Kaggle version has these key columns:
      - 'condition' or 'label': 0=no stress, 1=time pressure, 2=interruption
      - 'participant' or 'subject' or 'person': subject ID (1–25)
      - Remaining columns: HRV features (~34 numeric features)

    Some known HRV features in SWELL:
      MEAN_RR, MEDIAN_RR, SDRR, RMSSD, SDSD, SDRR_RMSSD,
      HR, pNN25, pNN50, SD1, SD2, KURT, SKEW,
      MEAN_REL_RR, MEDIAN_REL_RR, SDRR_REL_RR, RMSSD_REL_RR,
      SDSD_REL_RR, KURT_REL_RR, SKEW_REL_RR,
      VLF, LF, HF, VLF_PCT, LF_PCT, HF_PCT, LF_HF, LF_NU, HF_NU,
      TOTAL_POWER, (and _LOG, _SQRT variants)

    Returns:
      X:        np.array (N, TIME_STEPS, FEATURE_DIM)
      y:        np.array (N,) — labels 0, 1, 2
      subjects: np.array (N,) — participant IDs
      feature_names: list of feature column names
    """
    print("  Reading CSV …")
    df = pd.read_csv(csv_path)
    print(f"  Raw shape: {df.shape}")
    print(f"  Columns ({len(df.columns)}): {list(df.columns[:15])} …")

    # ─── Identify label column ───
    label_col = None
    for candidate in ['condition', 'label', 'Condition', 'Label', 'class', 'stress']:
        if candidate in df.columns:
            label_col = candidate
            break

    if label_col is None:
        # Try to find it by dtype
        for col in df.columns:
            unique_vals = df[col].dropna().unique()
            if len(unique_vals) <= 5 and set(unique_vals).issubset({0, 1, 2, 3}):
                label_col = col
                print(f"  Auto-detected label column: '{col}' with values {sorted(unique_vals)}")
                break

    if label_col is None:
        print(f"\n  ERROR: Cannot find label column!")
        print(f"  Available columns: {list(df.columns)}")
        print(f"  Please set label_col manually in the code.")
        raise ValueError("Label column not found")

    print(f"  Label column: '{label_col}'")
    print(f"  Label values: {sorted(df[label_col].dropna().unique())}")

    # ─── Identify subject/participant column ───
    subj_col = None
    for candidate in ['participant', 'subject', 'person', 'Participant',
                      'Subject', 'Person', 'pp', 'PP', 'id', 'ID']:
        if candidate in df.columns:
            subj_col = candidate
            break

    if subj_col is None:
        # Look for a column with ~25 unique integer values
        for col in df.columns:
            if col == label_col:
                continue
            nunique = df[col].nunique()
            if 10 <= nunique <= 30:
                subj_col = col
                print(f"  Auto-detected subject column: '{col}' ({nunique} unique)")
                break

    if subj_col is None:
        print("  WARNING: No subject column found! Creating synthetic subject IDs.")
        df['_subject_'] = np.arange(len(df)) % 25
        subj_col = '_subject_'
    else:
        print(f"  Subject column: '{subj_col}' ({df[subj_col].nunique()} subjects)")

    # ─── Extract feature columns (everything that's numeric except label & subject) ───
    skip_cols = {label_col, subj_col, '_subject_'}
    feature_cols = [
        col for col in df.columns
        if col not in skip_cols and pd.api.types.is_numeric_dtype(df[col])
    ]

    print(f"  Feature columns ({len(feature_cols)}): {feature_cols[:10]} …")

    # ─── Clean data ───
    df_clean = df[[subj_col, label_col] + feature_cols].dropna()

    # Remove any infinite values
    numeric_data = df_clean[feature_cols]
    inf_mask = np.isinf(numeric_data).any(axis=1)
    df_clean = df_clean[~inf_mask]

    print(f"  After cleaning: {len(df_clean)} rows")

    # ─── Extract arrays ───
    X_raw = df_clean[feature_cols].values.astype('float32')
    y_raw = df_clean[label_col].values

    # Encode labels to 0-indexed if not already
    le = LabelEncoder()
    y = le.fit_transform(y_raw)

    subjects = df_clean[subj_col].values
    # Make subject IDs integer
    if not np.issubdtype(subjects.dtype, np.integer):
        subj_le = LabelEncoder()
        subjects = subj_le.fit_transform(subjects)

    # ─── Normalize features (z-score) ───
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw).astype('float32')

    # ─── Reshape for HABC ───
    # SWELL-KW is tabular (one feature vector per sample, not a time series).
    # We create an artificial temporal dimension by tiling.
    TIME_STEPS = 10
    X_seq = np.tile(X_scaled[:, np.newaxis, :], (1, TIME_STEPS, 1))

    print(f"\n  Final shapes:")
    print(f"    X: {X_seq.shape}  (samples, time_steps, features)")
    print(f"    y: {y.shape}, classes: {sorted(np.unique(y))}")
    print(f"    subjects: {len(np.unique(subjects))} unique")

    return X_seq, y, subjects, feature_cols, le.classes_, TIME_STEPS


# ──────────── Actually load the data ────────────

print("=" * 60)
print("Loading SWELL-KW dataset …")
print("=" * 60)

raw_X, raw_y, raw_subjects, FEATURE_NAMES, CLASS_LABELS, TIME_STEPS = \
    load_swell_kw(csv_file)

FEATURE_DIM = raw_X.shape[2]
NUM_CLASSES = len(np.unique(raw_y))

# Build class names dict
CLASS_NAMES = {}
for i, label in enumerate(CLASS_LABELS):
    name_map = {0: 'No stress', 1: 'Time pressure', 2: 'Interruption'}
    if isinstance(label, (int, np.integer)):
        CLASS_NAMES[i] = name_map.get(int(label), f'Class {label}')
    else:
        CLASS_NAMES[i] = str(label)
CLASS_LIST = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]

print(f"\n{'='*60}")
print(f"  SWELL-KW Data Summary")
print(f"{'='*60}")
print(f"  Samples     : {len(raw_X)}")
print(f"  Time steps  : {TIME_STEPS}")
print(f"  Features    : {FEATURE_DIM}")
print(f"  Classes     : {NUM_CLASSES} → {CLASS_NAMES}")
print(f"  Class dist  : {dict(Counter(raw_y))}")
print(f"  Subjects    : {sorted(np.unique(raw_subjects))} ({len(np.unique(raw_subjects))} total)")

# Prepare inputs
X_cnn_all  = raw_X.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_all = raw_X.copy()
y_encoded  = to_categorical(raw_y, num_classes=NUM_CLASSES)

print(f"\n  CNN  input : {X_cnn_all.shape}")
print(f"  LSTM input : {X_lstm_all.shape}")
print(f"  Labels     : {y_encoded.shape}")



In [ ]:
# CELL 4: SANITY CHECK (optional — skip to save time)

print("\n" + "=" * 60)
print("SANITY CHECK: Training HABC on full SWELL-KW data")
print("=" * 60)

K.clear_session()
model_sanity = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
model_sanity.summary()

es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
history = model_sanity.fit(
    [X_cnn_all, X_lstm_all], y_encoded,
    epochs=100, batch_size=32, callbacks=[es], verbose=1
)

score = model_sanity.evaluate([X_cnn_all, X_lstm_all], y_encoded, verbose=0)
print(f"\nTraining-set accuracy: {score[1]*100:.2f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], lw=2)
ax1.set(title='Loss', xlabel='Epoch', ylabel='Loss'); ax1.grid(alpha=0.3)
ax2.plot(history.history['accuracy'], lw=2, color='green')
ax2.set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy'); ax2.grid(alpha=0.3)
plt.suptitle('HABC Training on SWELL-KW', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/sanity_training_curves.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# =============================================================================
# CELL 5: EXPERIMENT 1 — CROSS-SUBJECT GENERALIZATION
# =============================================================================

import json
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

print("\n" + "=" * 60)
print("EXPERIMENT 1: Cross-Subject Generalization — SWELL-KW")
print("=" * 60)

# ─── 1A: Leave-One-Subject-Out (LOSO) ───
print("\n--- 1A: LOSO ---")

logo = LeaveOneGroupOut()
unique_subjects = np.unique(raw_subjects)
n_folds = len(unique_subjects)
print(f"Running {n_folds} LOSO folds …\n")

loso_results = []

for fold, (tr_idx, te_idx) in enumerate(logo.split(raw_X, raw_y, raw_subjects)):
    te_subj = np.unique(raw_subjects[te_idx])[0]

    X_tr, X_te = raw_X[tr_idx], raw_X[te_idx]
    y_tr = to_categorical(raw_y[tr_idx], num_classes=NUM_CLASSES)
    y_te_raw = raw_y[te_idx]

    X_cnn_tr = X_tr.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
    X_lstm_tr = X_tr.copy()
    X_cnn_te = X_te.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
    X_lstm_te = X_te.copy()

    K.clear_session()
    model_f = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model_f.fit([X_cnn_tr, X_lstm_tr], y_tr,
                epochs=100, batch_size=32, verbose=0, callbacks=[es])

    y_pred = np.argmax(model_f.predict([X_cnn_te, X_lstm_te], verbose=0), axis=1)

    acc = accuracy_score(y_te_raw, y_pred)
    mf1 = f1_score(y_te_raw, y_pred, average='macro', zero_division=0)
    wf1 = f1_score(y_te_raw, y_pred, average='weighted', zero_division=0)

    loso_results.append({
        'subject': int(te_subj), 'accuracy': acc,
        'macro_f1': mf1, 'weighted_f1': wf1, 'n_test': len(te_idx)
    })
    print(f"  Fold {fold+1:2d}/{n_folds} | Subj {te_subj:2d} | "
          f"Acc={acc*100:6.2f}%  Macro-F1={mf1*100:6.2f}%  (n={len(te_idx)})")

loso_accs = [r['accuracy']    for r in loso_results]
loso_mf1  = [r['macro_f1']    for r in loso_results]
loso_wf1  = [r['weighted_f1'] for r in loso_results]

print(f"\n  LOSO RESULTS:")
print(f"    Accuracy    : {np.mean(loso_accs)*100:.2f}% ± {np.std(loso_accs)*100:.2f}%")
print(f"    Macro-F1    : {np.mean(loso_mf1)*100:.2f}% ± {np.std(loso_mf1)*100:.2f}%")
print(f"    Weighted-F1 : {np.mean(loso_wf1)*100:.2f}% ± {np.std(loso_wf1)*100:.2f}%")

# ─── 1B: Cross-Subject Split (70/30 by subject) ───
print("\n--- 1B: Cross-Subject Split (70/30, 5 random splits) ---")

gss = GroupShuffleSplit(n_splits=5, test_size=0.3, random_state=42)
cs_results = []

for split_i, (tr_idx, te_idx) in enumerate(gss.split(raw_X, raw_y, raw_subjects)):
    te_subs = sorted(np.unique(raw_subjects[te_idx]))

    X_tr, X_te = raw_X[tr_idx], raw_X[te_idx]
    y_tr = to_categorical(raw_y[tr_idx], num_classes=NUM_CLASSES)
    y_te_raw = raw_y[te_idx]

    K.clear_session()
    model_cs = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model_cs.fit([X_tr.reshape(-1,1,TIME_STEPS,FEATURE_DIM), X_tr.copy()],
                 y_tr, epochs=100, batch_size=32, verbose=0, callbacks=[es])

    y_pred = np.argmax(model_cs.predict(
        [X_te.reshape(-1,1,TIME_STEPS,FEATURE_DIM), X_te.copy()], verbose=0), axis=1)

    acc = accuracy_score(y_te_raw, y_pred)
    mf1 = f1_score(y_te_raw, y_pred, average='macro', zero_division=0)

    cs_results.append({'split': split_i+1, 'accuracy': acc, 'macro_f1': mf1,
                       'test_subjects': [int(s) for s in te_subs]})
    print(f"  Split {split_i+1}/5 | Test: {te_subs} | "
          f"Acc={acc*100:.2f}%  Macro-F1={mf1*100:.2f}%")

cs_accs = [r['accuracy'] for r in cs_results]
cs_mf1  = [r['macro_f1'] for r in cs_results]

print(f"\n  CROSS-SUBJECT RESULTS:")
print(f"    Accuracy : {np.mean(cs_accs)*100:.2f}% ± {np.std(cs_accs)*100:.2f}%")
print(f"    Macro-F1 : {np.mean(cs_mf1)*100:.2f}% ± {np.std(cs_mf1)*100:.2f}%")

# Save
exp1 = {
    "experiment": "Exp1_Cross_Subject", "dataset": "SWELL-KW",
    "LOSO": {"mean_acc": float(np.mean(loso_accs)),
             "std_acc": float(np.std(loso_accs)),
             "mean_macro_f1": float(np.mean(loso_mf1)),
             "std_macro_f1": float(np.std(loso_mf1)),
             "per_subject": loso_results},
    "Cross_Subject": {"mean_acc": float(np.mean(cs_accs)),
                      "std_acc": float(np.std(cs_accs)),
                      "mean_macro_f1": float(np.mean(cs_mf1)),
                      "per_split": cs_results}
}
with open(f"{RESULTS_DIR}/exp1_cross_subject.json", "w") as f:
    json.dump(exp1, f, indent=2, default=str)
print(f"\nSaved → {RESULTS_DIR}/exp1_cross_subject.json")

In [ ]:
# CELL 6: EXPERIMENT 3 — ROBUSTNESS TO NOISE & MISSING DATA

print("\n" + "=" * 60)
print("EXPERIMENT 3: Robustness — SWELL-KW")
print("=" * 60)

# Fixed cross-subject split for robustness testing
gss_r = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_r, te_r = next(gss_r.split(raw_X, raw_y, raw_subjects))

X_tr_r, X_te_r = raw_X[tr_r], raw_X[te_r]
y_tr_r = to_categorical(raw_y[tr_r], num_classes=NUM_CLASSES)
y_te_r_raw = raw_y[te_r]

X_cnn_tr_r  = X_tr_r.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_tr_r = X_tr_r.copy()
X_cnn_te_r  = X_te_r.reshape(-1, 1, TIME_STEPS, FEATURE_DIM)
X_lstm_te_r = X_te_r.copy()

# Train on clean data
K.clear_session()
model_robust = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model_robust.fit([X_cnn_tr_r, X_lstm_tr_r], y_tr_r,
                 epochs=100, batch_size=32, verbose=0, callbacks=[es])

# Clean baseline
y_pred_clean = np.argmax(model_robust.predict(
    [X_cnn_te_r, X_lstm_te_r], verbose=0), axis=1)
clean_acc = accuracy_score(y_te_r_raw, y_pred_clean)
print(f"\n  Clean test accuracy: {clean_acc*100:.2f}%")

# ── 3A: Gaussian Noise ──
print("\n--- 3A: Gaussian Noise ---")

def add_noise(X, snr_db, seed=42):
    rng = np.random.RandomState(seed)
    sp = np.mean(X ** 2)
    if sp == 0: return X.copy()
    np_  = sp / (10 ** (snr_db / 10))
    return (X + rng.normal(0, np.sqrt(np_), X.shape)).astype('float32')

snr_levels = [30, 20, 10, 5, 0]
noise_results = {'clean': float(clean_acc)}

for snr in snr_levels:
    Xn = add_noise(X_te_r, snr)
    yp = np.argmax(model_robust.predict(
        [Xn.reshape(-1,1,TIME_STEPS,FEATURE_DIM), Xn.copy()], verbose=0), axis=1)
    acc = accuracy_score(y_te_r_raw, yp)
    noise_results[f'snr_{snr}'] = float(acc)
    print(f"  SNR={snr:3d}dB → Acc={acc*100:.2f}%")

# ── 3B: Missing Data ──
print("\n--- 3B: Missing Data ---")

def add_missing(X, pct, seed=42):
    rng = np.random.RandomState(seed)
    Xm = X.copy()
    Xm[rng.random(X.shape) < pct] = 0.0
    return Xm

miss_levels = [0.0, 0.05, 0.10, 0.15, 0.20, 0.30]
miss_results = {}

for pct in miss_levels:
    Xm = add_missing(X_te_r, pct)
    yp = np.argmax(model_robust.predict(
        [Xm.reshape(-1,1,TIME_STEPS,FEATURE_DIM), Xm.copy()], verbose=0), axis=1)
    acc = accuracy_score(y_te_r_raw, yp)
    miss_results[f'{pct*100:.0f}%'] = float(acc)
    print(f"  Missing={pct*100:5.1f}% → Acc={acc*100:.2f}%")

# ── Plots ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(snr_levels, [noise_results[f'snr_{s}']*100 for s in snr_levels],
         'bo-', lw=2, ms=8)
ax1.axhline(clean_acc*100, color='g', ls='--', alpha=.7,
            label=f'Clean ({clean_acc*100:.1f}%)')
ax1.set(xlabel='SNR (dB)', ylabel='Accuracy (%)',
        title='Robustness to Noise')
ax1.invert_xaxis(); ax1.legend(); ax1.grid(alpha=0.3); ax1.set_ylim(0,105)

miss_x = [0,5,10,15,20,30]
ax2.plot(miss_x, [miss_results[f'{p}%']*100 for p in miss_x],
         'rs-', lw=2, ms=8)
ax2.set(xlabel='Missing (%)', ylabel='Accuracy (%)',
        title='Robustness to Missing Data')
ax2.grid(alpha=0.3); ax2.set_ylim(0,105)

plt.suptitle('Robustness — SWELL-KW', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp3_robustness.png", dpi=300, bbox_inches='tight')
plt.show()

exp3 = {"experiment": "Exp3_Robustness", "dataset": "SWELL-KW",
        "clean_acc": float(clean_acc),
        "noise": noise_results, "missing": miss_results}
with open(f"{RESULTS_DIR}/exp3_robustness.json", "w") as f:
    json.dump(exp3, f, indent=2)
print(f"Saved → exp3_robustness.json + exp3_robustness.png")



In [ ]:
# CELL 7: EXPERIMENT 4 — STATISTICAL SIGNIFICANCE

print("\n" + "=" * 60)
print("EXPERIMENT 4: Statistical Significance — SWELL-KW")
print("=" * 60)
print("Running LOSO for baselines …\n")

baseline_cfgs = {
    "CNN only":    {"build": build_cnn_only,    "mode": "cnn"},
    "BiLSTM only": {"build": build_bilstm_only, "mode": "lstm"},
}

sig_folds = {"HABC": loso_accs}

for bl_name, bl_cfg in baseline_cfgs.items():
    print(f"  Running LOSO for: {bl_name}")
    bl_accs = []

    for fold, (tr_idx, te_idx) in enumerate(logo.split(raw_X, raw_y, raw_subjects)):
        X_tr, X_te = raw_X[tr_idx], raw_X[te_idx]
        y_tr = to_categorical(raw_y[tr_idx], num_classes=NUM_CLASSES)
        y_te_raw = raw_y[te_idx]

        if bl_cfg["mode"] == "cnn":
            tr_in = [X_tr.reshape(-1,1,TIME_STEPS,FEATURE_DIM)]
            te_in = [X_te.reshape(-1,1,TIME_STEPS,FEATURE_DIM)]
        else:
            tr_in = [X_tr.copy()]
            te_in = [X_te.copy()]

        K.clear_session()
        m = bl_cfg["build"](NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
        es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
        m.fit(tr_in, y_tr, epochs=100, batch_size=32, verbose=0, callbacks=[es])
        yp = np.argmax(m.predict(te_in, verbose=0), axis=1)
        bl_accs.append(accuracy_score(y_te_raw, yp))

        if (fold+1) % 5 == 0:
            print(f"    fold {fold+1}/{n_folds} done")

    sig_folds[bl_name] = bl_accs
    print(f"  {bl_name} mean: {np.mean(bl_accs)*100:.2f}% ± {np.std(bl_accs)*100:.2f}%\n")

# Run tests
print(f"\n{'='*75}")
print(f"  {'Comparison':<28} {'HABC':>8} {'Base':>8} {'t':>7} {'p(t)':>10} {'W':>7} {'p(W)':>10}")
print(f"{'='*75}")

habc_arr = np.array(sig_folds["HABC"])
sig_results = {}

for bl_name in baseline_cfgs:
    bl_arr = np.array(sig_folds[bl_name])
    t_s, p_t = sp_stats.ttest_rel(habc_arr, bl_arr)
    try:
        w_s, p_w = sp_stats.wilcoxon(habc_arr, bl_arr)
    except:
        w_s, p_w = float('nan'), float('nan')

    sig_results[bl_name] = {
        'habc_mean': float(np.mean(habc_arr)),
        'baseline_mean': float(np.mean(bl_arr)),
        't_stat': float(t_s), 'p_ttest': float(p_t),
        'w_stat': float(w_s), 'p_wilcoxon': float(p_w),
        'significant': bool(p_t < 0.05)
    }
    pt = f"{p_t:.2e}" if p_t < 0.001 else f"{p_t:.4f}"
    pw = f"{p_w:.2e}" if (not np.isnan(p_w) and p_w < 0.001) else f"{p_w:.4f}"
    print(f"  HABC vs {bl_name:<18} {np.mean(habc_arr)*100:>7.2f}% "
          f"{np.mean(bl_arr)*100:>7.2f}% {t_s:>7.3f} {pt:>10} {w_s:>7.1f} {pw:>10}")

print(f"{'='*75}")

exp4 = {"experiment": "Exp4_Significance", "dataset": "SWELL-KW",
        "folds": {k: [float(v) for v in vs] for k, vs in sig_folds.items()},
        "tests": sig_results}
with open(f"{RESULTS_DIR}/exp4_significance.json", "w") as f:
    json.dump(exp4, f, indent=2)
print(f"\nSaved → exp4_significance.json")



In [ ]:
# CELL 8: EXPERIMENT 8 — CONFUSION MATRIX

print("\n" + "=" * 60)
print("EXPERIMENT 8: Confusion Matrix — SWELL-KW")
print("=" * 60)

# Use same split as robustness experiment
K.clear_session()
model_cm = build_habc_model(NUM_CLASSES, TIME_STEPS, FEATURE_DIM)
es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
model_cm.fit([X_cnn_tr_r, X_lstm_tr_r], y_tr_r,
             epochs=100, batch_size=32, verbose=0, callbacks=[es])

y_pred_cm = np.argmax(model_cm.predict([X_cnn_te_r, X_lstm_te_r], verbose=0), axis=1)
y_true_cm = y_te_r_raw

# Raw
cm = confusion_matrix(y_true_cm, y_pred_cm)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_LIST, yticklabels=CLASS_LIST,
            linewidths=0.5, linecolor='lightgray')
ax.set(xlabel='Predicted', ylabel='True',
       title='Confusion Matrix — HABC on SWELL-KW')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp8_confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues', ax=ax,
            xticklabels=CLASS_LIST, yticklabels=CLASS_LIST,
            linewidths=0.5, linecolor='lightgray', vmin=0, vmax=1)
ax.set(xlabel='Predicted', ylabel='True',
       title='Normalized Confusion Matrix — HABC on SWELL-KW')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp8_confusion_matrix_norm.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved → exp8_confusion_matrix.png + exp8_confusion_matrix_norm.png")


In [ ]:
# CELL 9: EXPERIMENT 10 — CLASS-WISE METRICS

print("\n" + "=" * 60)
print("EXPERIMENT 10: Class-Wise Metrics — SWELL-KW")
print("=" * 60)

report = classification_report(y_true_cm, y_pred_cm,
                               target_names=CLASS_LIST,
                               output_dict=True, zero_division=0)

# Table
print(f"\n{'='*65}")
print(f"  {'Class':<20} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
print(f"{'='*65}")
for cls in CLASS_LIST:
    r = report[cls]
    print(f"  {cls:<20} {r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}% "
          f"{r['f1-score']*100:>9.2f}% {r['support']:>9.0f}")
print(f"  {'-'*60}")
for avg in ['macro avg', 'weighted avg']:
    r = report[avg]
    print(f"  {avg.title():<20} {r['precision']*100:>9.2f}% {r['recall']*100:>9.2f}% "
          f"{r['f1-score']*100:>9.2f}% {r['support']:>9.0f}")
print(f"{'='*65}")

overall = accuracy_score(y_true_cm, y_pred_cm)
print(f"\n  Overall Accuracy: {overall*100:.2f}%")

print(f"\n--- Full Report ---")
print(classification_report(y_true_cm, y_pred_cm,
                            target_names=CLASS_LIST, digits=4, zero_division=0))

# Per-class F1 bar chart
fig, ax = plt.subplots(figsize=(8, 5))
f1_vals = [report[c]['f1-score']*100 for c in CLASS_LIST]
colors = ['#4CAF50' if v >= 70 else '#FF9800' if v >= 45 else '#F44336' for v in f1_vals]
ax.bar(CLASS_LIST, f1_vals, color=colors, edgecolor='gray', lw=0.5)
ax.axhline(np.mean(f1_vals), color='blue', ls='--', alpha=0.7,
           label=f'Mean F1 = {np.mean(f1_vals):.1f}%')
ax.set(ylabel='F1-Score (%)', title='Per-Class F1-Score — HABC on SWELL-KW')
ax.legend(); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/exp10_per_class_f1.png", dpi=300, bbox_inches='tight')
plt.show()

exp10 = {"experiment": "Exp10_ClassWise", "dataset": "SWELL-KW",
         "overall_accuracy": float(overall), "report": report}
with open(f"{RESULTS_DIR}/exp10_classwise.json", "w") as f:
    json.dump(exp10, f, indent=2)
print(f"Saved → exp10_classwise.json + exp10_per_class_f1.png")



In [ ]:
# CELL 10: FINAL SUMMARY

print("\n" + "=" * 70)
print("  FINAL SUMMARY — ALL SWELL-KW EXPERIMENTS COMPLETE")
print("=" * 70)

print(f"""
┌───────────────────────────────────────────────────────────────────────┐
│  Dataset: SWELL-KW │ Samples: {len(raw_X):,} │ Classes: {NUM_CLASSES} │ Subjects: {len(np.unique(raw_subjects))}       │
├───────────────────────────────────────────────────────────────────────┤
│  Exp 1: Cross-Subject                                                │
│    LOSO Accuracy:      {np.mean(loso_accs)*100:6.2f}% ± {np.std(loso_accs)*100:.2f}%                         │
│    LOSO Macro-F1:      {np.mean(loso_mf1)*100:6.2f}% ± {np.std(loso_mf1)*100:.2f}%                         │
│    Cross-Sub Accuracy: {np.mean(cs_accs)*100:6.2f}% ± {np.std(cs_accs)*100:.2f}%                         │
├───────────────────────────────────────────────────────────────────────┤
│  Exp 3: Robustness                                                   │
│    Clean: {clean_acc*100:.2f}%  SNR=10dB: {noise_results.get('snr_10',0)*100:.2f}%  Missing=20%: {miss_results.get('20%',0)*100:.2f}%    │
├───────────────────────────────────────────────────────────────────────┤
│  Exp 4: Statistical Significance                                     │""")
for bn, sr in sig_results.items():
    mark = "✓" if sr['significant'] else "✗"
    print(f"│    HABC vs {bn:<18} p={sr['p_ttest']:.4f} {mark}                         │")
print(f"""├───────────────────────────────────────────────────────────────────────┤
│  Exp 8:  Confusion matrices saved ✓                                  │
│  Exp 10: Class-wise metrics saved ✓  (Overall: {overall*100:.2f}%)               │
├───────────────────────────────────────────────────────────────────────┤
│  All files → {RESULTS_DIR}/                      │
└───────────────────────────────────────────────────────────────────────┘
""")

master = {
    "model": "HABC (CNN + BiLSTM + Attention)",
    "dataset": "SWELL-KW",
    "num_samples": int(len(raw_X)),
    "num_classes": NUM_CLASSES,
    "num_features": FEATURE_DIM,
    "num_subjects": int(len(np.unique(raw_subjects))),
    "experiments": [
        "Exp1: Cross-Subject (LOSO + Cross-Subject Split)",
        "Exp3: Robustness (Noise + Missing)",
        "Exp4: Statistical Significance (t-test + Wilcoxon)",
        "Exp8: Confusion Matrix (raw + normalized)",
        "Exp10: Class-Wise Metrics + chart",
    ],
    "output_files": [
        "sanity_training_curves.png",
        "exp3_robustness.png",
        "exp8_confusion_matrix.png",
        "exp8_confusion_matrix_norm.png",
        "exp10_per_class_f1.png",
        "exp1_cross_subject.json",
        "exp3_robustness.json",
        "exp4_significance.json",
        "exp10_classwise.json",
        "master_summary.json",
    ]
}
with open(f"{RESULTS_DIR}/master_summary.json", "w") as f:
    json.dump(master, f, indent=2)

print("ALL SWELL-KW EXPERIMENTS COMPLETE!")
print(f"Check: {RESULTS_DIR}/")